In [3]:
from importlib.metadata import version

print(f"datasketch version:  {version("datasketch")}")
print(f"jieba version:  {version("jieba")}")

datasketch version:  1.7.0
jieba version:  0.42.1


In [2]:
import json
import jieba
from tqdm import tqdm
from datasketch import MinHash, MinHashLSH
from concurrent.futures import ProcessPoolExecutor, as_completed 

In [3]:
minhash = MinHash(num_perm=128)
print("插入前：", minhash.hashvalues, minhash.hashvalues.shape)

for word in ["我", "学", "AI"]:
    minhash.update(word.encode("utf-8"))
print("插入后：", minhash.hashvalues, minhash.hashvalues.shape)

插入前： [4294967295 4294967295 4294967295 4294967295 4294967295 4294967295
 4294967295 4294967295 4294967295 4294967295 4294967295 4294967295
 4294967295 4294967295 4294967295 4294967295 4294967295 4294967295
 4294967295 4294967295 4294967295 4294967295 4294967295 4294967295
 4294967295 4294967295 4294967295 4294967295 4294967295 4294967295
 4294967295 4294967295 4294967295 4294967295 4294967295 4294967295
 4294967295 4294967295 4294967295 4294967295 4294967295 4294967295
 4294967295 4294967295 4294967295 4294967295 4294967295 4294967295
 4294967295 4294967295 4294967295 4294967295 4294967295 4294967295
 4294967295 4294967295 4294967295 4294967295 4294967295 4294967295
 4294967295 4294967295 4294967295 4294967295 4294967295 4294967295
 4294967295 4294967295 4294967295 4294967295 4294967295 4294967295
 4294967295 4294967295 4294967295 4294967295 4294967295 4294967295
 4294967295 4294967295 4294967295 4294967295 4294967295 4294967295
 4294967295 4294967295 4294967295 4294967295 4294967295 4

In [4]:
! head -3 data/baike_qa/baike_qa_train.json

{"qid": "qid_5982723620932473219", "category": "教育/科学-理工学科-地球科学", "title": "人站在地球上为什么没有头朝下的感觉 ", "desc": "", "answer": "地球上重力作用一直是指向球心的，因此\r\n只要头远离球心，人们就回感到头朝上。"}
{"qid": "qid_5679706523376347837", "category": "娱乐-宠物", "title": "我的小baby", "desc": "我的小baby-辛巴。温顺可爱的，两个月大的小家伙，第一次养狗，该注意什么呢？求指教～[爱你]", "answer": "勤洗澡，养成好的卫生习惯"}
{"qid": "qid_6610724023825624555", "category": "娱乐-度假旅游", "title": "请问这起交通事故是谁的责任居多?小车和摩托车发生事故，在无红绿灯 ", "desc": "小车和摩托车发生事故，在无红绿灯的十字路口，小停车看看左右，在觉得安全的情况下刹车慢慢以时速10公里左右的速度靠右行驶过路口，好没有出到十字路口正中时，被左边突然快速行驶过来的摩托车撞在车头前，摩托车主摔到膝盖和檫伤脸部，请问这起交通事故是谁的责任居多。如果双方都有责任的话，大概各占几成？~\r", "answer": "通过没有信号控制的十字路口，应该减速慢性，让右边的车先行，按你说的，摩托车好像在汽车的左边，所以严格来说可能摩托车全责。当然还要看汽车是否证照齐全，是否饮酒等。具体由交警调查后认定。"}


In [5]:
! wc -l data/baike_qa/baike_qa_train.json

1425170 data/baike_qa/baike_qa_train.json


In [6]:
def process_dialog(idx): # 多进程这里不要传入dialogs，会变慢很多
    words = jieba.lcut(dialogs[idx])
    minhash = MinHash(num_perm=num_perm)
    for word in words:
        minhash.update(word.encode("utf-8"))
    return idx, minhash

# MinHash 和 LSH快速去重

In [7]:
# 参数配置
num_perm = 128 # minhash的函数个数
threshold = 0.7 #相似度阈值（可调整）
max_workers = 12 # 最大worker数

# 读取数据
fd = open("../data/baike_qa/baike_qa_train.json")
dialogs = []
for line in tqdm(fd, desc="Load Data"):
    info = json.loads(line)
    dialogs.append(info["title"] + info["answer"])

Load Data: 1425170it [00:14, 98615.11it/s] 


In [8]:
total_dialogs = len(dialogs)
print(total_dialogs)

1425170


In [9]:
# 初始化minhashLSH
lsh = MinHashLSH(threshold=threshold, num_perm=num_perm)

# 为每条对话创建 MinHash 并加入LSH
minhashes = [None] * total_dialogs

# 使用多进程生成 minhash vector
with ProcessPoolExecutor(max_workers=max_workers) as executor:
    futures = [executor.submit(process_dialog, idx) for idx in range(total_dialogs)]
    for future in tqdm(as_completed(futures), total=total_dialogs, desc="Generate Hash Vectors"):
        idx, minhash = future.result()
        lsh.insert(f"dialog_{idx}", minhash) # 插入LSH的时候就已经分桶了
        minhashes[idx] = minhash

Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Building prefix dict from the default dictionary ...
Building prefix dict from the default dictionary ...
Building prefix dict from the default dictionary ...
Building prefix dict from the default dictionary ...
Building prefix dict from the default dictionary ...
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model from cache /tmp/jieba.cache
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model from cache /tm

In [12]:
minhashes[-1].hashvalues

array([  7771666, 281527510,  47429436,  67529222, 145144577,  12745232,
       129589284,  40389565, 103058441, 364624756,  41653339, 113697518,
        99670005,  60243568,   3987473,  27628122,  10842363,  47330701,
        27015704, 123896294,  12060161,  92422174,  78132356,  73403669,
        11120616,  19049009,   3651271,  51466846,  57999906,  20656453,
        32332222, 129034955,  91024506,   2727417, 201795309, 113150351,
       195799941,  12642616,  33768288,  12833290,  91011190,  17881277,
       227001007,  10994249,  11139440, 145584451, 166118373, 180300913,
         5377335,  47496329, 218676373,  69986434,  31534853,  19251391,
        44220935,  40923525,  55646121,  57245472, 132333677,  78143114,
        61011754, 120414426,  30596415,  18422277, 201393534,  10394921,
        36345193, 211179662, 255782418,  45606839, 424731769, 275528182,
        57171742,  18739391,   9422124,  19582943,  38626927,  57001644,
       353966956,  19042281,  58265887,  82303665, 

In [14]:
unique_dialogs = [] # 去重后的文档
seen = set() # 标记已处理的文档
print_count = 0
max_print = 3
for idx in tqdm(range(total_dialogs), desc="Deduplicate docs"):
    if idx in seen:
        continue

    minhash = minhashes[idx]
    result = lsh.query(minhash) # 查询相似文档
    similar_idxs = [int(r.split("_")[1]) for r in result if r != f"dialog_{idx}"] # 找当前文档的相似文档索引
    if not similar_idxs: # 没有相似的文档
        unique_dialogs.append(dialogs[idx])
    else: # 有相似文档
        unique_dialogs.append(dialogs[idx])
        seen.update(similar_idxs) # 标记相似文档为已处理

        # 打印相似对话信息
        if print_count < max_print:
            print(f"\n输入： \n '{dialogs[idx]}' \n 与以下文档相似: (阈值={threshold}): \n")
            for sim_idx in similar_idxs[:max_print]:
                print(f"\n 输出 {sim_idx}: \n '{dialogs[sim_idx]}'")
            print("*" * 100)
            print_count += 1
    seen.add(idx)

# 打印去重结果的统计信息
print("\n==== 去重结果 ====")
print(f"原始文档数量: {total_dialogs}")
print(f"去重后的文档数量: {len(unique_dialogs)}")
print(f"重复的文档数量: {total_dialogs - len(unique_dialogs)}")

Deduplicate docs:   0%|          | 3446/1425170 [00:00<01:19, 17829.16it/s]


输入： 
7、栗皮紧肤--取栗子的内果皮，捣成末状，与蜂蜜均匀搅拌，涂于面部，能使脸部光洁、富有弹性。' 蛋汁里，使二者混合均匀。平日可将此面膜储存在冰箱里，一周做1-2次就可以让肌肤紧实，改善毛孔粗大，促进皮肤的光滑细致。 毕现，毛孔也明显起来，往日的平滑细致不复存在了。 ，保湿就成为更必要的急救法门了。所以，许多标榜能缩小毛孔的产品，同时也有保湿的成分。 肌肤变得平滑柔嫩，但水杨酸所使用的浓度多寡更与成效息息相关。在医生帮患者进行去角质换肤时，浓度多控制在3%－10％；而保养品则多限制在0．2%~1．5％间。比如Clinique倩碧的洁肤水、SK－II的精致换肤霜、欧莱雅的细致毛孔精华露与Neutrogena露得清的毛孔细致精华霜都含有不同浓度比例的水杨酸，帮助肌肤清理角质层，全面改善毛孔和肤质。 
 与以下文档相似: (阈值=0.7): 


 输出 1196761: 
7、栗皮紧肤--取栗子的内果皮，捣成末状，与蜂蜜均匀搅拌，涂于面部，能使脸部光洁、富有弹性。'鸡蛋汁里，使二者混合均匀。平日可将此面膜储存在冰箱里，一周做1-2次就可以让肌肤紧实，改善毛孔粗大，促进皮肤的光滑细致。 毛孔也明显起来，往日的平滑细致不复存在了。 ，保湿就成为更必要的急救法门了。所以，许多标榜能缩小毛孔的产品，同时也有保湿的成分。 肌肤变得平滑柔嫩，但水杨酸所使用的浓度多寡更与成效息息相关。在医生帮患者进行去角质换肤时，浓度多控制在3%－10％；而保养品则多限制在0．2%~1．5％间。比如Clinique倩碧的洁肤水、SK－II的精致换肤霜、欧莱雅的细致毛孔精华露与Neutrogena露得清的毛孔细致精华霜都含有不同浓度比例的水杨酸，帮助肌肤清理角质层，全面改善毛孔和肤质。 

 输出 733208: 
7、栗皮紧肤--取栗子的内果皮，捣成末状，与蜂蜜均匀搅拌，涂于面部，能使脸部光洁、富有弹性。'鸡蛋汁里，使二者混合均匀。平日可将此面膜储存在冰箱里，一周做1-2次就可以让肌肤紧实，改善毛孔粗大，促进皮肤的光滑细致。 毕现，毛孔也明显起来，往日的平滑细致不复存在了。 ，保湿就成为更必要的急救法门了。所以，许多标榜能缩小毛孔的产品，同时也有保湿的成分。 肌肤变得平滑柔嫩，但水杨酸所使用的浓度多寡更与成效息息相关。在医生帮患者进行去角质换肤时，浓度多控制在3%－10％；而保养品则多限制

Deduplicate docs: 100%|██████████| 1425170/1425170 [00:42<00:00, 33465.96it/s]


==== 去重结果 ====
原始文档数量: 1425170
去重后的文档数量: 1367173
重复的文档数量: 57997
